# Day 3 — PySpark Practice Questions: Date & Time Functions

| # | Difficulty | Topic |
|---|-----------|-------|
| 1 | 🟢 Easy    | Find slow deliveries (datediff) |
| 2 | 🟢 Easy    | Monthly revenue totals (date_trunc + groupBy) |
| 3 | 🟡 Medium  | New customers — first order in last 90 days (spark_min + filter) |

In [ ]:
import os
os.environ['JAVA_HOME']           = 'C:/Program Files/DBeaver/jre'
os.environ['PYSPARK_PYTHON']        = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit,
    to_date, datediff, date_trunc, date_format,
    year, month, dayofmonth,
    date_add, date_sub,
    months_between, add_months,
    current_date,
    min as spark_min, max as spark_max,
    sum as spark_sum, count, avg,
    round as spark_round,
)
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType,
)

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('Day3_PySpark_PQ') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

In [ ]:
# Shared dataset — same orders as SQL
data = [
    (1, 1001, '2026-05-01', '2026-05-05', 250.00),
    (2, 1002, '2026-05-03', '2026-05-14', 189.50),
    (3, 1001, '2026-05-10', '2026-05-17', 310.00),
    (4, 1003, '2026-05-15', '2026-05-23', 95.00),
    (5, 1004, '2026-05-20', '2026-05-21', 540.00),
    (6, 1002, '2026-06-01', '2026-06-03', 220.00),
    (7, 1005, '2026-06-05', '2026-06-19', 75.00),
    (8, 1003, '2026-06-10', '2026-06-12', 430.00),
    (9, 1006, '2026-06-12', '2026-06-14', 180.00),
    (10,1007, '2026-06-15', '2026-06-16', 99.00),
    (11,1001, '2025-01-05', '2025-01-07', 500.00),
    (12,1002, '2025-03-12', '2025-03-14', 320.00),
    (13,1003, '2025-06-20', '2025-07-01', 150.00),
    (14,1004, '2025-09-08', '2025-09-10', 780.00),
    (15,1005, '2025-11-25', '2025-11-30', 210.00),
    (16,1001, '2025-12-01', '2025-12-15', 640.00),
    (17,1002, '2025-12-20', '2025-12-22', 110.00),
    (18,1004, '2026-01-10', '2026-01-12', 920.00),
    (19,1001, '2026-02-14', '2026-02-16', 275.00),
    (20,1003, '2026-03-01', '2026-03-04', 360.00),
    (21,1002, '2026-03-18', '2026-03-20', 490.00),
    (22,1005, '2026-04-05', '2026-04-07', 130.00),
]

schema = StructType([
    StructField('order_id',    IntegerType(), False),
    StructField('customer_id', IntegerType(), False),
    StructField('order_date',  StringType(),  False),
    StructField('ship_date',   StringType(),  True),
    StructField('amount',      DoubleType(),  False),
])

df_orders = spark.createDataFrame(data, schema) \
    .withColumn('order_date', to_date(col('order_date'), 'yyyy-MM-dd')) \
    .withColumn('ship_date',  to_date(col('ship_date'),  'yyyy-MM-dd'))

df_orders.printSchema()
df_orders.show(5)

---
# Question 1 🟢 — Slow Deliveries

## Concept Warm-up

In [ ]:
# Warm-up 1: datediff(end_col, start_col) — end goes FIRST
from pyspark.sql.functions import datediff, col

df_orders.select(
    'order_id', 'order_date', 'ship_date',
    datediff(col('ship_date'), col('order_date')).alias('delivery_days')
).show(5)

In [ ]:
# Warm-up 2: WRONG order — what happens when you put start first
df_orders.select(
    'order_id',
    datediff(col('order_date'), col('ship_date')).alias('WRONG_negative_days')
).show(3)

In [ ]:
# Warm-up 3: filter with withColumn then filter
df_check = df_orders \
    .withColumn('delivery_days', datediff(col('ship_date'), col('order_date'))) \
    .filter(col('delivery_days') > 5)
print('Orders with >5 delivery days:', df_check.count())
df_check.select('order_id', 'order_date', 'ship_date', 'delivery_days').show()

## Problem

The logistics team wants to review orders where delivery took **more than 7 days**.  
Return `order_id`, `customer_id`, `order_date`, `ship_date`, and `delivery_days`, sorted slowest first.

In [ ]:
# YOUR CODE HERE
df_slow = None

# df_slow.show()

---
# Question 2 🟢 — Monthly Revenue Totals

## Concept Warm-up

In [ ]:
# Warm-up 1: date_trunc maps all dates to first of month
from pyspark.sql.functions import date_trunc

df_orders.select(
    'order_date',
    date_trunc('month', col('order_date')).alias('month_start')
).distinct().orderBy('month_start').show()

In [ ]:
# Warm-up 2: date_format produces a sortable string label
from pyspark.sql.functions import date_format

df_orders.select(
    'order_date',
    date_format(col('order_date'), 'yyyy-MM').alias('ym'),
    date_format(col('order_date'), 'MMM yyyy').alias('label')
).show(5)

In [ ]:
# Warm-up 3: groupBy + agg pattern
from pyspark.sql.functions import spark_sum, count

df_orders \
    .groupBy(date_trunc('month', col('order_date')).alias('month_ts')) \
    .agg(count('*').alias('num_orders')) \
    .orderBy('month_ts') \
    .show()

## Problem

Build a **monthly revenue report** with: `month` (as `'yyyy-MM'`), `total_revenue`, `num_orders`, `avg_order_value`,  
ordered chronologically from oldest to most recent.

In [ ]:
# YOUR CODE HERE
df_monthly = None

# df_monthly.show()

---
# Question 3 🟡 — New Customers (First Order in Last 90 Days)

## Concept Warm-up

In [ ]:
# Warm-up 1: current_date() and date_sub to get 90-day cutoff
from pyspark.sql.functions import current_date, date_sub

spark.range(1).select(
    current_date().alias('today'),
    date_sub(current_date(), 90).alias('ninety_days_ago')
).show()

In [ ]:
# Warm-up 2: spark_min to find first order date per customer
from pyspark.sql.functions import min as spark_min

df_first = df_orders.groupBy('customer_id') \
    .agg(spark_min('order_date').alias('first_order_date'))
df_first.orderBy('first_order_date').show()

In [ ]:
# Warm-up 3: filter a grouped result by date comparison
# Orders placed in the last 90 days:
df_recent_rows = df_orders.filter(
    col('order_date') >= date_sub(current_date(), 90)
)
print('Recent order rows:', df_recent_rows.count())
df_recent_rows.select('order_id', 'customer_id', 'order_date').show()

In [ ]:
# Warm-up 4: datediff to compute days since first order
df_first_with_age = df_first.withColumn(
    'days_since_first',
    datediff(current_date(), col('first_order_date'))
)
df_first_with_age.orderBy('days_since_first').show()

## Problem

The growth team defines a **new customer** as someone whose **first-ever order** was placed within the last 90 days.  
Return `customer_id`, `first_order_date`, and `days_since_first_order`, sorted most-recent first.

In [ ]:
# YOUR CODE HERE
df_new_customers = None

# df_new_customers.show()